# Q1–Q2 · Truflation vs official CPI

In [1]:
%load_ext autoreload
%autoreload 2
import logging
import pandas as pd
import plotly.io as pio
from IPython.display import display, Markdown

import pipeline as pl
import charts as ch

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.precision", 2, "display.width", 140)
logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", force=True)

## Q1 · Data pipeline
Truflation via TRUF.NETWORK (no key) · CPI via FRED, checked against BLS · runs on demand or daily (`python pipeline.py --daily-at 15:00`).

In [2]:
res = pl.run_pipeline()          # same call the daily job / a dashboard 'refresh' button makes
res.meta

2026-10-21 22:07:03,992 WARNING DATA CHECK: official CPI missing for ['2025-10']


{'run_at_utc': '2026-10-21T14:07:04+00:00',
 'seconds': 22.2,
 'truflation_source': 'truflation_gateway',
 'truflation_last_date': '2026-10-21',
 'truflation_last_value': 3.038,
 'cpi_last_month': '2026-09',
 'cpi_last_yoy': 3.74,
 'next_cpi_release': '2026-11-10',
 'pce_last_month': '2026-08',
 'pce_last_yoy': 3.42,
 'next_pce_release': '2026-10-29',
 'bls_crosscheck': True,
 'issues': ["official CPI missing for ['2025-10']"]}

In [3]:
truf, cpi, monthly = res.truflation, res.cpi, res.monthly
summary = pd.DataFrame({
    "Truflation (daily)": [truf.index.min().date(), truf.index.max().date(), len(truf), truf["truflation_yoy"].iloc[-1]],
    "Official CPI YoY (monthly)": [cpi["cpi_yoy"].first_valid_index().date(), cpi["cpi_yoy"].last_valid_index().date(),
                                   cpi["cpi_yoy"].notna().sum(), cpi["cpi_yoy"].dropna().iloc[-1]],
}, index=["first", "last", "observations", "latest YoY %"]).T
display(summary)
display(cpi.tail(14)[["cpi_nsa", "cpi_sa", "cpi_yoy", "core_yoy", "cpi_mom_sa", "core_mom_sa"]])

,first,last,observations,latest YoY %
Truflation (daily),2010-01-01,2026-10-21,6138,3.04
Official CPI YoY (monthly),2011-01-01,2026-09-01,188,3.74


,cpi_nsa,cpi_sa,cpi_yoy,core_yoy,cpi_mom_sa,core_mom_sa
date,,,,,,
2025-08-01,323.98,323.29,2.92,3.11,0.35,0.31
2025-09-01,324.80,324.25,3.01,3.02,0.30,0.22
2025-10-01,NaN,NaN,NaN,NaN,NaN,NaN
2025-11-01,324.12,325.06,2.74,2.63,NaN,NaN
2025-12-01,324.05,326.03,2.68,2.64,0.30,0.23
2026-01-01,325.25,326.59,2.39,2.50,0.17,0.30
2026-02-01,326.79,327.46,2.41,2.46,0.27,0.22
2026-03-01,330.21,330.29,3.26,2.60,0.87,0.20
2026-04-01,333.02,332.41,3.81,2.75,0.64,0.38


In [4]:
print("Issues flagged:", res.issues or "none")

bls = res.bls                    # BLS copy fetched (and cached) by the pipeline run above
if bls is None:
    print("FRED vs BLS: BLS unavailable this run (e.g. keyless daily quota) - FRED used alone")
else:
    rec = cpi[["cpi_nsa", "core_nsa"]].join(bls[["cpi_nsa", "core_nsa"]], rsuffix="_bls", how="inner").dropna()
    print(f"FRED vs BLS: {len(rec)} overlapping months, max abs diff headline = "
          f"{(rec.cpi_nsa - rec.cpi_nsa_bls).abs().max():.4f}, core = {(rec.core_nsa - rec.core_nsa_bls).abs().max():.4f}")

days = monthly["truf_days"].dropna()
print(f"Truflation days per month: min {days.iloc[:-1].min():.0f}, median {days.median():.0f} "
      f"(current month so far: {days.iloc[-1]:.0f})")

Issues flagged: ["official CPI missing for ['2025-10']"]
FRED vs BLS: 200 overlapping months, max abs diff headline = 0.0000, core = 0.0000
Truflation days per month: min 28, median 31 (current month so far: 21)


## Q2 · Visualisation
Interactive version: [https://us-inflation-monitor.onrender.com/](https://us-inflation-monitor.onrender.com/)

In [5]:
ch.fig_yoy_comparison(truf, cpi, res.recessions, res.releases).show()

In [6]:
ch.fig_monthly_divergence(monthly, res.recessions).show()

In [7]:
ch.fig_scatter(monthly).show()

In [8]:
stats = ch.regime_stats(monthly)
display(stats.style.format(ch.STATS_FORMAT))      # spread statistics to 3 d.p.

,months,corr (levels),corr (1m changes),mean spread (pp),MAE (pp),max |spread| (pp)
Full sample,188,0.94,0.84,+0.087,0.653,3.544
2010–2019 (low inflation),108,0.93,0.78,+0.078,0.316,1.247
2020–2022 (COVID & surge),36,0.98,0.92,+1.264,1.417,3.544
2023–2024 (disinflation),24,0.95,0.91,-0.696,0.701,1.313
2025– (tariffs),20,0.28,0.56,-1.042,1.042,2.210


In [9]:
t = truf["truflation_yoy"]
cur_month = t.index.max().to_period("M")
mtd = t[t.index.to_period("M") == cur_month]
last_cpi_month = cpi["cpi_yoy"].last_valid_index()
display(Markdown(f'''
| | |
|---|---|
| Truflation latest ({t.index.max():%d-%b-%Y}) | **{t.iloc[-1]:.2f}%** |
| Truflation {cur_month.strftime("%b-%y")} month-to-date average ({len(mtd)} days) | **{mtd.mean():.2f}%** |
| Truflation 30 days ago | {t.asof(t.index.max() - pd.Timedelta(days=30)):.2f}% |
| Last official CPI YoY ({last_cpi_month:%b-%y}) | **{cpi.loc[last_cpi_month, "cpi_yoy"]:.2f}%** (core {cpi.loc[last_cpi_month, "core_yoy"]:.2f}%) |
| Truflation − CPI spread, last print | {monthly.loc[last_cpi_month, "spread"]:+.3f}pp |
| Next CPI release | **{res.meta["next_cpi_release"]}** ({(cur_month).strftime("%b-%y")} data) |
'''))


| | |
|---|---|
| Truflation latest (21-Oct-2026) | **3.04%** |
| Truflation Oct-26 month-to-date average (21 days) | **2.94%** |
| Truflation 30 days ago | 2.39% |
| Last official CPI YoY (Sep-26) | **3.74%** (core 2.56%) |
| Truflation − CPI spread, last print | -1.349pp |
| Next CPI release | **2026-11-10** (Oct-26 data) |


**Takeaway:** tracks CPI's direction (≈0.84), not its level.

### Extra: core CPI, PCE, core PCE

In [10]:
ch.fig_yoy_comparison_grid(truf, cpi, res.recessions, res.releases).show()

In [11]:
ch.fig_monthly_divergence_grid(monthly, res.recessions).show()

In [12]:
ch.fig_scatter_grid(monthly).show()

In [13]:
stats_all = ch.regime_stats_all(monthly)
display(stats_all.style.format(ch.STATS_FORMAT))

last = {}
for key, (col, label) in pl.MEASURES.items():
    d = monthly[f"spread_{key}"].last_valid_index()
    last[label] = {"last print": d.strftime("%b-%y"), "official YoY %": round(monthly.loc[d, col], 3),
                   "Truflation month-avg %": round(monthly.loc[d, "truf_avg"], 3),
                   "spread (pp)": monthly.loc[d, f"spread_{key}"]}
display(pd.DataFrame(last).T.style.format({"official YoY %": "{:.3f}", "Truflation month-avg %": "{:.3f}",
                                           "spread (pp)": "{:+.3f}"}))
print(f"Next PCE release: {res.meta['next_pce_release']}  |  next CPI release: {res.meta['next_cpi_release']}")

,last print,official YoY %,Truflation month-avg %,spread (pp)
Headline CPI,Sep-26,3.738,2.389,-1.349
Core CPI,Sep-26,2.564,2.389,-0.176
PCE,Aug-26,3.419,2.289,-1.130
Core PCE,Aug-26,3.008,2.289,-0.719


Next PCE release: 2026-10-29  |  next CPI release: 2026-11-10


Headline measures track Truflation best; core measures much worse.